In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote
import os
import time


def crawl_jobkorea():
    results = []

    keyword = "데이터분석"
    page = 1

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.jobkorea.co.kr/"
    }

    encoded_keyword = quote(keyword)

    search_url = (
        "https://www.jobkorea.co.kr/Search/"
        f"?stext={encoded_keyword}&tabType=recruit&Page_No={page}"
    )

    response = requests.get(search_url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    title_links = soup.select("a[href*='/Recruit/GI_Read/']")

    seen_links = set()

    for title_tag in title_links:
        recruit = title_tag.get_text(" ", strip=True)

        if recruit == "":
            continue

        href = title_tag.get("href", "")

        if href.startswith("http"):
            detail_url = href
        else:
            detail_url = "https://www.jobkorea.co.kr" + href

        if detail_url in seen_links:
            continue

        seen_links.add(detail_url)

        company = ""
        detail = ""

        try:
            detail_response = requests.get(detail_url, headers=headers, timeout=10)
            detail_response.raise_for_status()

            detail_soup = BeautifulSoup(detail_response.text, "html.parser")

            company_tag = detail_soup.find(
                "h2",
                class_=lambda x: x
                and "font-medium" in x
                and "text-[20px]" in x
            )

            if company_tag:
                company = company_tag.get_text(" ", strip=True)

            if company == "":
                h2_tags = detail_soup.find_all("h2")

                for tag in h2_tags:
                    text = tag.get_text(" ", strip=True)

                    if (
                        text != ""
                        and text != recruit
                        and "회원가입" not in text
                        and "로그인" not in text
                        and len(text) <= 50
                    ):
                        company = text
                        break

            full_text = detail_soup.get_text(" ", strip=True)

            region = ""

            region_keywords = [
                "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산",
                "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
            ]

            if "근무지주소" in full_text:
                region_part = full_text.split("근무지주소", 1)[1]
                words = region_part.split()

                if len(words) >= 2:
                    region = words[0] + " " + words[1]
                elif len(words) >= 1:
                    region = words[0]

            if region == "" and "근무지역" in full_text:
                region_part = full_text.split("근무지역", 1)[1]
                words = region_part.split()

                if len(words) >= 2:
                    region = words[0] + " " + words[1]
                elif len(words) >= 1:
                    region = words[0]

            if region == "":
                for word in region_keywords:
                    if word in full_text:
                        region = word
                        break

            career = ""

            if "경력무관" in full_text:
                career = "경력무관"
            elif "신입" in full_text and "경력" in full_text:
                career = "신입·경력"
            elif "신입" in full_text:
                career = "신입"
            elif "경력" in full_text:
                career = "경력"

            education = ""

            education_keywords = [
                "학력무관",
                "고졸↑",
                "초대졸↑",
                "대졸↑",
                "석사↑",
                "박사↑",
                "고졸",
                "초대졸",
                "대졸",
                "석사",
                "박사"
            ]

            for word in education_keywords:
                if word in full_text:
                    education = word
                    break

            employment = ""

            employment_keywords = [
                "정규직",
                "계약직",
                "프리랜서",
                "인턴",
                "아르바이트",
                "파견직",
                "위촉직",
                "교육생"
            ]

            for word in employment_keywords:
                if word in full_text:
                    employment = word
                    break

            detail_items = []

            if region != "":
                detail_items.append(region)

            if career != "":
                detail_items.append(career)

            if education != "":
                detail_items.append(education)

            if employment != "":
                detail_items.append(employment)

            detail = ", ".join(detail_items)

        except Exception:
            company = ""
            detail = ""

        results.append({
            "Site": "Job_Korea",
            "Col_Company": company,
            "Col_Recruit": recruit,
            "Col_detail": detail,
            "Col_url": detail_url
        })

        time.sleep(0.5)

    df = pd.DataFrame(
        results,
        columns=["Site", "Col_Company", "Col_Recruit", "Col_detail", "Col_url"]
    )

    return df


df_jobkorea = crawl_jobkorea()

os.makedirs("data_tmp", exist_ok=True)

df_jobkorea.to_csv(
    "data_tmp/data_jobkorea.csv",
    index=False,
    encoding="utf-8-sig"
)

print("data_tmp/data_jobkorea.csv 저장 완료")
print(df_jobkorea.shape)

df_jobkorea.head()


data_tmp/data_jobkorea.csv 저장 완료
(26, 5)


,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Job_Korea,한국모니터링시스템,용접 자동화 설비 S/W 개발 (비전과 제어),"서울 영등포구, 신입·경력, 대졸, 정규직",https://www.jobkorea.co.kr/Recruit/GI_Read/496...
1,Job_Korea,헬스앤바이오㈜,[케어팟] 생활가전 하드웨어 개발경력,"서울 강남구, 신입·경력, 초대졸, 정규직",https://www.jobkorea.co.kr/Recruit/GI_Read/496...
2,Job_Korea,㈜휴먼교육센터,[국비최대무료/취업연계/기숙사제공]AI인공지능/빅데이터/풀스택/부트캠프,"충남 천안시, 신입·경력, 학력무관, 인턴",https://www.jobkorea.co.kr/Recruit/GI_Read/495...
3,Job_Korea,웍스피어(유),[웍스피어] 데이터분석가 (마케팅 DA) (4년이상),"대한민국 서울특별시, 신입·경력, 학력무관, 정규직",https://www.jobkorea.co.kr/Recruit/GI_Read/495...
4,Job_Korea,콘센트릭스서비스코리아,데이터분석태깅/기획,"서울 강남구, 신입·경력, 학력무관, 정규직",https://www.jobkorea.co.kr/Recruit/GI_Read/496...
